# Exercises for session 4 (solution)

The data that you will use come from the [replication package](https://github.com/RicardoGabriel/The-Political-Costs-of-Austerity) to Ricardo Duque
Gabriel, Mathias Klein, and Ana Sofia Pessoa: *The Political Costs of Austerity*,
Review of Economics and Statistics, 2023.

> **Note:**
> 
> Please commit every time you solve one of the exercises. An example commit message
> could be `"Solution to question 1"`. Feel free to commit more than once per
> exercise if solving it requires multiple complicated steps.
>
> Push every now and then and switch to somebody else's machine.

## Using the `pathlib` library

---
### Question 1

Assign the path of the current directory to a variable `this_dir`. Verify that the type
of the variable is `pathlib.PosixPath` or `pathlib.WindowsPath`. Display the absolute
path of the directory.

In [ ]:

from pathlib import Path

this_dir = Path()
type(this_dir)

In [ ]:

this_dir.resolve()

---
### Question 2

In the `original_data` directory, there is a file called `Data_Elections.dta`. Assign the path of
this file to a variable `data_file`. The type of the variable should be 
`pathlib.PosixPath` or `pathlib.WindowsPath`. 

- When creating the `Path` object, do not use absolute paths.
- Display the absolute path of the `data_file`.

In [ ]:

data_file = this_dir / "original_data" / "Data_Elections.dta"
data_file.resolve()

---
### Question 3 (spend 5 Minutes max on it, else consider as Bonus!)

Using only the objects `this_dir` and `data_file` along with their methods, get the relative path to `data_file` as seen from `this_dir`. Display the relative path.

> Note: We have not seen this in the screencast; you are on your own with your favourite
> search engine.

In [ ]:

data_file.resolve().relative_to(this_dir.resolve())

## pandas

---
### Question 4

- Import the `pandas` library as `pd`. 
- Set the options so you use "modern" Pandas as described in the first screencast. 
- Set the plotting backend to `plotly`.


In [ ]:

import pandas as pd

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = False
pd.options.plotting.backend = "plotly"


---
### Question 5

Read the file `Data_Elections.dta` into a `pd.DataFrame` object called `data`. Use the
`data_file` object for doing so.

You are likely to get an error when doing so. Find out
how to fix it. *(Hint: The error message is even more explicit than usually in Python;
you may want to have a carefully look at it despite its length. If working in VS Code,
make sure that you can see the entire message by selecting from the view options at the
bottom of the cell output)*

In [ ]:

data = pd.read_stata(data_file, convert_categoricals=False)

---
### Question 6

Familiarise yourself with the dataset. E.g., yo may want to look the column names, the
shape, some rows, data types ...

In [ ]:

data.columns

In [ ]:

data.shape

In [ ]:

data

In [ ]:

data.dtypes

---
### Question 7

We had to discard some information when reading the dataset. Luckily, we can access all
of it using a low-level `pd.StataReader` object. This will allow us to look at all meta
information that is stored with the dataset in Stata format.

In [ ]:
data_info = pd.io.stata.StataReader(data_file)

Look at the various `labels` attributes of the `pd.StataReader` object. Explain what
they do. Can you explain now why the error occurred when reading the dataset?

In [ ]:

data_info.variable_labels()

In [ ]:

data_info.value_labels()

In [ ]:

data_info.data_label


- `data_label` is a string describing the dataset. In this case, it is empty.
- `variable_labels` is a dictionary mapping the column names we have in our dataset to
  verbose descriptions.
- `value_labels` is a nested dictionary with column names as keys on the outer level.
  The inner dictionaries map the numeric values in the dataset to verbose descriptions.
  It is filled only for `ElectionType`. This is very much how `pd.Categorical` is
  stored internally, although we cannot see the numerical values in that case (which is
  a good thing! Far too easy to run numerical calculations with unordered categorical
  variables in Stata accidentally).
- The error occurred because the `National + Regional` label is repeated for two
  different values of `ElectionType`. While `value_labels` are supposed to be a
  bijection, Stata does not enforce this. Pandas will check it when converting the
  `value_labels` dictionary to a `pd.Categorical` object with the value labels as
  categories. Since categories need to be unique, 5 and 7 would be merged and
  information would be lost. Hence the error.


---
### Question 8

Look at the structure of `data` again. Would you keep all of

- `Country` and `cid`?
- `Nuts_id`,  `Name`, and `id`?

Why or why not?



- No reason to keep `cid`. Countries are very few and we have the country names. Most
  certainly, these are no official country codes, so we won't need them for merging.
- `Nuts_id` is a unique identifier for the regions and they look quite official. While
  we do have the names, they may easily differ across datasets (unicode characters, 
  parentheses, ...) or not be present in all datasets. `id` looks like a numerical code
  for `Nuts_id`, but we better verify.

---
### Question 9

Make sure that we can safely drop `cid` and `id` from the dataset by finding out the
unique combinations of the sets of variables from the previous question.

> Note: The `.unique()` method only works on Series, you'll need to find something else.

In [ ]:

data[["Country", "cid"]].drop_duplicates()

In [ ]:

(
    len(data[["Country", "cid"]].drop_duplicates())
    == len(data[["Country"]].drop_duplicates())
    == len(data[["cid"]].drop_duplicates())
)

In [ ]:

data[["Nuts_id", "Name", "id"]].drop_duplicates()

In [ ]:

(
    len(data[["Nuts_id", "Name", "id"]].drop_duplicates())
    == len(data["Nuts_id"].drop_duplicates())
    == len(data[["Name"]].drop_duplicates())
    == len(data[["id"]].drop_duplicates())
)

---
### Question 10

Now clean the data using the **functional data management** approach from the
screencasts, following the three rules:

1. Start with an empty DataFrame
2. Touch every variable just once
3. Touch it with a pure function

Below, there is one header per column of the raw dataset, followed by an
empty code cell and an empty markdown cell. Implement your treatment of the
column in the code cell and note the reasoning behind it in the markdown
cell. The solutions notebook contains our reasoning in those places. For
each column, decide for yourself how to treat it:

- What is a good name for it? It should be informative and consistent with
  the other columns.
- Which dtype should it have? Use what you found out in Questions 5-9 about
  the columns' contents, variable labels, and value labels.
- Is a plain `.astype()` call enough, or does the conversion warrant a small
  (pure!) helper function?
- Or should the column not be part of the cleaned dataset at all? Note that
  there is no explicit "drop" step in this approach: You simply never assign
  such a column to the new DataFrame, i.e., there is nothing to do in the
  respective cell.

> **Hint:** One helper function will be useful for several columns. Call it
> `_round_to_uint32` and define it in the cell where you first need it.

Start by creating the empty DataFrame `df`, reusing the index of `data`.

In [ ]:

df = pd.DataFrame(index=data.index)

#### `Country`

In [ ]:

df["country"] = data["Country"].astype(pd.CategoricalDtype())


The country names are strings from a small set of values, so a
categorical dtype is the natural representation. The new name fixes
the capitalisation.

#### `cid`

In [ ]:

# Nothing to do.


Question 9 showed that `cid` carries no information beyond `Country`.
Dropping it simply means never assigning it to `df`.

#### `Nuts_id`

In [ ]:

df["nuts_id"] = data["Nuts_id"].astype(pd.CategoricalDtype())


A string column with a limited set of values again, hence categorical.
Only the capitalisation of the name changes.

#### `Name`

In [ ]:

df["nuts_name"] = data["Name"].astype(pd.CategoricalDtype())


The same conversion as before. The new name `nuts_name` makes clear
which entity the name refers to.

#### `Year`

In [ ]:

df["year"] = data["Year"].astype(pd.Int16Dtype())


Years are integers, so the floating point dtype coming from Stata is
not appropriate. The nullable `Int16Dtype` is enough for calendar
years.

#### `ElectionType`

In [ ]:


def _convert_election_cats(raw_sr, labels_dict):
    election_cats = labels_dict.copy()
    election_cats[5] += " A"
    election_cats[7] += " B"
    sr = raw_sr.astype(pd.Int8Dtype()).astype(pd.CategoricalDtype())
    return sr.cat.rename_categories(election_cats)


df["election_type"] = _convert_election_cats(
    raw_sr=data["ElectionType"],
    labels_dict=data_info.value_labels()["ElectionType"],
)


- This conversion clearly warrants its own pure helper function.
- Get Stata's `value_labels` and fix the duplicated `National + Regional`
  label (see Question 7). The `.copy()` is important, else we would modify
  the original dictionary from `data_info`.
- The raw column first needs to be converted to a nullable integer dtype to
  match its contents, then to a `category`; then we call
  `.rename_categories()` with the fixed dictionary.

#### `EligibleVoters`

In [ ]:


def _round_to_uint32(sr):
    return sr.round().astype(pd.UInt32Dtype())


df["number_eligible_voters"] = _round_to_uint32(data["EligibleVoters"])


- Counts of voters should be non-negative integers, but Stata stores them as
  floats. Some of the values are numerically too far away from integers for
  pandas to convert them directly, so we need to explicitly round first. In
  cases where it would be crucial to get the correct integer, we may want to
  investigate deeper if that happens (e.g., we expect integers 1 to 7, but
  may have a 1.4 in there). Here, it does not matter given the size of the
  electorates and likely measurement error.
- Several more columns below are counts of votes / voters, so this helper
  will be reused a couple of times.

#### `Valid`

In [ ]:

df["number_valid_votes"] = _round_to_uint32(data["Valid"])


Another count of votes, so `_round_to_uint32` is reused.

#### `HHI`

In [ ]:

df["number_parties_effective"] = data["HHI"]


The variable label tells us that `HHI` is the effective number of parties (a
transformation of the Herfindahl-Hirschman index), so we give it a name in
line with the other columns. Despite being a "number of", it is a fractional
measure, so we keep it as a float.

#### `Far_Right`

In [ ]:

df["number_votes_far_right"] = _round_to_uint32(data["Far_Right"])


A count of votes, treated in the same way as the previous ones.

#### `Far_Left`

In [ ]:

df["number_votes_far_left"] = _round_to_uint32(data["Far_Left"])


The same as for `Far_Right`.

#### `Far_Right_share`

In [ ]:

df["share_votes_far_right"] = data["Far_Right_share"]


Shares are fractional by nature, so the float dtype is already
appropriate. The column just receives a name consistent with the
other columns.

#### `Far_Left_share`

In [ ]:

df["share_votes_far_left"] = data["Far_Left_share"]


The same as for `Far_Right_share`.

#### `Far_share`

In [ ]:

df["share_votes_far_any"] = data["Far_share"]


The same again; the new name makes explicit that it combines far
right and far left parties.

#### `Turnout`

In [ ]:

df["share_voter_turnout"] = data["Turnout"]


A share as well; the new name makes that explicit.

#### `F0Far_Incumbent`

In [ ]:

df["number_votes_far_any_incumbent"] = _round_to_uint32(data["F0Far_Incumbent"])


A count of votes again. The variable label tells us that it refers to
votes for far right or far left parties that are part of the
government.

#### `left`

In [ ]:


def _pm_party_orientation(pm_party_left):
    sr = pm_party_left.astype(pd.Int8Dtype()).astype(pd.CategoricalDtype())
    return sr.cat.rename_categories(
        {0: "Other", 1: "Left-leaning"},
    )


df["pm_party_orientation"] = _pm_party_orientation(data["left"])


- Even though we often say "X is a dummy variable for whatever", the better
  representation usually are categorical variables.
- This will make plotting etc. easier and more consistent.
- Decent statistical packages handle these out-of-the-box, too.

#### `id`

In [ ]:

# Nothing to do.


Like `cid`, the column `id` only duplicates information, this time
from `Nuts_id` (see Question 9). Hence it never gets assigned to
`df`.

Look at the result of your work:

In [ ]:
df

---
### Question 11

Every column of `df` was created by a pure function of the raw data. Hence,
collecting your work in a single pure function `clean_data(raw, metadata)` is
a matter of copy & paste. We will need this function in order to leave the
notebook world behind, see Question 15. Write it and replace `data` by its
cleaned version.

In [ ]:


def clean_data(raw, metadata):
    df = pd.DataFrame(index=raw.index)
    df["country"] = raw["Country"].astype(pd.CategoricalDtype())
    df["nuts_id"] = raw["Nuts_id"].astype(pd.CategoricalDtype())
    df["nuts_name"] = raw["Name"].astype(pd.CategoricalDtype())
    df["year"] = raw["Year"].astype(pd.Int16Dtype())
    df["election_type"] = _convert_election_cats(
        raw_sr=raw["ElectionType"],
        labels_dict=metadata.value_labels()["ElectionType"],
    )
    df["number_eligible_voters"] = _round_to_uint32(raw["EligibleVoters"])
    df["number_valid_votes"] = _round_to_uint32(raw["Valid"])
    df["number_parties_effective"] = raw["HHI"]
    df["number_votes_far_right"] = _round_to_uint32(raw["Far_Right"])
    df["number_votes_far_left"] = _round_to_uint32(raw["Far_Left"])
    df["share_votes_far_right"] = raw["Far_Right_share"]
    df["share_votes_far_left"] = raw["Far_Left_share"]
    df["share_votes_far_any"] = raw["Far_share"]
    df["share_voter_turnout"] = raw["Turnout"]
    df["number_votes_far_any_incumbent"] = _round_to_uint32(raw["F0Far_Incumbent"])
    df["pm_party_orientation"] = _pm_party_orientation(raw["left"])
    return df


data = clean_data(raw=data, metadata=data_info)
data


- `clean_data` touches each variable exactly once: either a plain
  `.astype()` call inline, or a call to one of the pure helper functions
  from above.
- Because we start from an empty DataFrame, columns we do not want (`cid`,
  `id`) are simply never assigned; there is no need for an explicit
  `.drop()` call.
- Renaming happens for free: the new column name is just the key we assign
  to on the left-hand side.

---
### Question 12

Summarise the cleaned data in a similar way as above. Now also look at summary
statistics, including value counts of categorical variables. Do you notice anything
when doing the latter? If so, will you have to be careful for interpreting descriptive
statistics?

In [ ]:

data.columns

In [ ]:

data.shape

In [ ]:

data

In [ ]:

data.dtypes

In [ ]:

data.describe()

In [ ]:

data["country"].value_counts()

In [ ]:

data["nuts_id"].value_counts().unique()

In [ ]:

data["election_type"].value_counts()

In [ ]:

data["pm_party_orientation"].value_counts()


- Value counts for NUTS ids show that we have a balanced panel
- Apparently always last election results are used, see query below
- Need to be extremely careful when doing descriptives, because all other
  variables are filled.

In [ ]:

data.query("nuts_id == 'SE33'")

---
### Question 13

Make three plots of mean vote shares by year, across all NUTS regions and elections.
- far right share
- far left share
- far share (any)

Don't worry about whether these plots make a lot of sense (i.e., any weighting with
electorate size or the like).

In [ ]:

only_elections = data[data["election_type"].notna()]
only_elections.shape

In [ ]:

only_elections.groupby("year")["share_votes_far_right"].mean().plot()

In [ ]:

only_elections.groupby("year")["share_votes_far_left"].mean().plot()

In [ ]:

only_elections.groupby("year")["share_votes_far_any"].mean().plot()

---
### Question 14

Make three scatterplots of vote shares by year, across all NUTS regions and elections.

- far right share
- far left share
- far share (any)

Colour the dots using the country.

In [ ]:

only_elections.plot.scatter(x="year", y="share_votes_far_right", color="country")

In [ ]:

only_elections.plot.scatter(x="year", y="share_votes_far_left", color="country")

In [ ]:

only_elections.plot.scatter(x="year", y="share_votes_far_any", color="country")

---
### Question 15

Wrap `clean_data` into a pytask task. Create a file
`task_clean_election_data.py` containing a task function
`task_clean_election_data(data_file=..., produces=BLD / "election_results.pkl")`
that reads the raw Stata data, calls `clean_data`, and writes the result to
`produces` with `to_pickle`. Move (or copy) your `clean_data` and its helper
functions into that file.

Run `pytask` from the command line and verify that `bld/election_results.pkl`
is created and matches the `data` you already have in this notebook.


The file needs the following on top of your `clean_data` and its helper
functions from Question 11:

```python
from pathlib import Path

import pandas as pd

ORIGINAL_DATA = Path(__file__).parent / "original_data"
BLD = Path(__file__).parent / "bld"


def task_clean_election_data(
    data_file=ORIGINAL_DATA / "Data_Elections.dta",
    produces=BLD / "election_results.pkl",
):
    data = pd.read_stata(data_file, convert_categoricals=False)
    data_info = pd.io.stata.StataReader(data_file)
    clean = clean_data(raw=data, metadata=data_info)
    clean.to_pickle(produces)
```

Running `pytask` should report one succeeding task and create
`bld/election_results.pkl`. You can verify the contents with, e.g.,
`pd.read_pickle(this_dir / "bld" / "election_results.pkl").equals(data)`.